# PCA-dimension comparison: P0/P2/P4 versus P0/P2

This experiment compares two PT-coefficient emulators side by side using the same completed training dataset, identical cosmology split, identical preprocessing strategy, and identical MLP architecture. The only experimental factors are the predicted multipoles and the number of retained PCA components.

**Dataset:** `training_data/velocileptors_pt_lhs_v2_restricted/pt_tables/`

**PCA dimensions:** 8, 12, 16, 24, 32, 48.

For every multipole set and PCA dimension, the notebook reports:
1. PCA-only holdout reconstruction error (compression loss, before the NN).
2. NN holdout PT-table error (compression plus learned mapping).
3. Reconstructed-spectrum holdout error after applying the same standard-basis nuisance coefficients analytically.

The P0/P2 model selects ell=0,2 from each original three-multipole file. No training-data regeneration is needed.

Run from the project checkout in the configured NERSC environment. This notebook deliberately does not initialize CLASS or evaluate a new cosmology with TheoryManager: spectrum-level errors are measured by contracting the held-out true and predicted PT tables with the same standard-basis nuisance vector. This isolates the emulator error.

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor

np.set_printoptions(precision=6, suppress=True)
RANDOM_SEED = 20261112
PCA_DIMENSIONS = [8, 12, 16, 24, 32, 48]
TEST_SIZE = 0.20

# Keep these identical for every run.
MLP_KWARGS = dict(
    hidden_layer_sizes=(128, 128),
    activation="tanh",
    solver="adam",
    alpha=1e-4,
    learning_rate_init=3e-4,
    max_iter=3000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=100,
    tol=1e-6,
    random_state=RANDOM_SEED,
    verbose=False,
)

## 1. Load the common training dataset

The source files contain all three multipoles with shape (3, 117, 19). We load them once and construct the two target datasets from the same ordered array.

In [ ]:
DATA_ROOT = Path("./training_data")
RUN_NAME = "velocileptors_pt_lhs_v2_restricted"
RUN_DIR = DATA_ROOT / RUN_NAME
TABLE_DIR = RUN_DIR / "pt_tables"

if not TABLE_DIR.is_dir():
    raise FileNotFoundError(f"PT-table directory not found: {TABLE_DIR.resolve()}")

with open(RUN_DIR / "run_config.json") as stream:
    run_config = json.load(stream)

COSMOLOGY_PARAMETERS = [
    "omega_b", "omega_cdm", "sigma8_cb", "n_s",
    "alpha_s", "N_ur", "w0_fld", "wa_fld",
]
SOURCE_ELL = np.array([0, 2, 4], dtype=int)
SOURCE_SHAPE = (3, 117, 19)

table_files = sorted(TABLE_DIR.glob("cosmo_*.npz"))
if not table_files:
    raise FileNotFoundError(f"No cosmo_*.npz files found in {TABLE_DIR}")

records, tables = [], []
native_k_reference = None
component_labels_reference = None

for path in table_files:
    with np.load(path, allow_pickle=False) as saved:
        required = {"pt_table", "k_native", "ells", "component_labels",
                    "cosmology_json", "metadata_json"}
        missing = required.difference(saved.files)
        if missing:
            raise KeyError(f"{path.name} is missing keys: {sorted(missing)}")
        table = np.asarray(saved["pt_table"], dtype=np.float64)
        k_native = np.asarray(saved["k_native"], dtype=np.float64)
        ells = np.asarray(saved["ells"], dtype=int)
        labels = saved["component_labels"].astype(str).tolist()
        cosmo = json.loads(str(saved["cosmology_json"].item()))
        metadata = json.loads(str(saved["metadata_json"].item()))

    if table.shape != SOURCE_SHAPE:
        raise ValueError(f"{path.name}: table shape {table.shape}, expected {SOURCE_SHAPE}")
    if not np.isfinite(table).all():
        raise ValueError(f"{path.name}: non-finite PT-table values")
    if k_native.shape != (SOURCE_SHAPE[1],) or not np.isfinite(k_native).all():
        raise ValueError(f"{path.name}: invalid native k coordinate")
    if not np.all(np.diff(k_native) > 0):
        raise ValueError(f"{path.name}: native k must be strictly increasing")
    if not np.array_equal(ells, SOURCE_ELL):
        raise ValueError(f"{path.name}: unexpected multipoles {ells}")
    if len(labels) != SOURCE_SHAPE[2]:
        raise ValueError(f"{path.name}: expected 19 component labels")

    if native_k_reference is None:
        native_k_reference = k_native.copy()
        component_labels_reference = labels
    else:
        if not np.allclose(k_native, native_k_reference, rtol=0, atol=1e-12):
            raise ValueError(f"{path.name}: native k grid differs")
        if labels != component_labels_reference:
            raise ValueError(f"{path.name}: component-label ordering changed")

    if not all(name in cosmo for name in COSMOLOGY_PARAMETERS):
        raise ValueError(f"{path.name}: missing cosmological coordinates")
    xrow = [float(cosmo[name]) for name in COSMOLOGY_PARAMETERS]
    if not np.isfinite(xrow).all():
        raise ValueError(f"{path.name}: non-finite cosmological coordinates")

    file_index = int(path.stem.split("_")[-1])
    if int(metadata.get("cosmology_index", -1)) != file_index:
        raise ValueError(f"{path.name}: filename and metadata indices disagree")

    records.append({
        "cosmology_index": file_index,
        **dict(zip(COSMOLOGY_PARAMETERS, xrow)),
    })
    tables.append(table)

order = np.argsort([r["cosmology_index"] for r in records])
X_all = pd.DataFrame(records).iloc[order].reset_index(drop=True)
Y_024 = np.stack(tables, axis=0)[order]

if X_all["cosmology_index"].duplicated().any():
    raise ValueError("Duplicate cosmology indices found")

X = X_all[COSMOLOGY_PARAMETERS].to_numpy(dtype=np.float64)
Y_02 = Y_024[:, :2, :, :].copy()

n_cosmo = len(X)
print("Run:", RUN_NAME)
print("Configured cosmologies:", run_config.get("n_cosmologies"))
print("Loaded cosmologies:", n_cosmo)
print("Cosmology input shape:", X.shape)
print("P0/P2/P4 table shape:", Y_024.shape)
print("P0/P2 table shape:", Y_02.shape)
print("Native k range:", (native_k_reference.min(), native_k_reference.max()))

## 2. Make one shared train/holdout split

We split row indices once and reuse them for both target datasets and every PCA dimension. Thus, differences between the two emulator families are not caused by different holdout cosmologies.

In [ ]:
all_indices = np.arange(n_cosmo)
train_idx, valid_idx = train_test_split(
    all_indices, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

X_train_raw, X_valid_raw = X[train_idx], X[valid_idx]
x_scaler = StandardScaler().fit(X_train_raw)
X_train = x_scaler.transform(X_train_raw)
X_valid = x_scaler.transform(X_valid_raw)

print("Training cosmologies:", len(train_idx))
print("Holdout cosmologies:", len(valid_idx))
print("Same split will be used for P0/P2/P4 and P0/P2.")

## 3. Define preprocessing, metrics, and exact standard-basis contraction

Component scaling is fitted on the training partition only. PCA is also fitted on training cosmologies only. The spectrum comparison uses the same 11 standard-basis nuisance coefficients as the working P0/P2 cosmology-comparison cell; no physical-to-standard conversion or growth-rate extraction is performed.

In [ ]:
from desilike.theories.galaxy_clustering.full_shape import (
    _velocileptors_combine_bias_terms_spectrum2_poles,
)
from src.theory import K_FID

FIDUCIAL_NUISANCE = {
    "b1": 2.0, "b2": 0.0, "bs": 0.0, "b3": 0.0,
    "alpha0": 0.0, "alpha2": 0.0, "alpha4": 0.0, "alpha6": 0.0,
    "sn0": 0.0, "sn2": 0.0, "sn4": 0.0,
}
STANDARD_PARAMETER_ORDER = [
    "b1", "b2", "bs", "b3",
    "alpha0", "alpha2", "alpha4", "alpha6",
    "sn0", "sn2", "sn4",
]
standard_pars = np.asarray(
    [FIDUCIAL_NUISANCE[name] for name in STANDARD_PARAMETER_ORDER],
    dtype=float,
)
K_FID = np.asarray(K_FID, dtype=float)

def table_error_summary(truth, prediction):
    diff = prediction - truth
    denom = max(float(np.sqrt(np.mean(truth**2))), 1e-30)
    return {
        "relative_rms": float(np.sqrt(np.mean(diff**2)) / denom),
        "rmse": float(np.sqrt(np.mean(diff**2))),
        "max_abs": float(np.max(np.abs(diff))),
    }

def spectra_from_tables(table_batch):
    """Contract a batch of native PT tables using fixed standard nuisance values."""
    spectra = []
    for table in table_batch:
        native_pk = np.asarray(
            _velocileptors_combine_bias_terms_spectrum2_poles(
                np.asarray(table, dtype=float), standard_pars
            ),
            dtype=float,
        )
        expected = (table.shape[0], len(native_k_reference))
        if native_pk.shape != expected:
            raise ValueError(
                f"Unexpected contraction shape {native_pk.shape}; expected {expected}"
            )
        output_pk = np.vstack([
            np.interp(K_FID, native_k_reference, row)
            for row in native_pk
        ])
        spectra.append(output_pk)
    return np.asarray(spectra)

print("Standard nuisance vector:", standard_pars)
print("Output k samples:", len(K_FID))

## 4. Run the PCA-dimension sweep for both emulator families

For each target family, the loop fits component scaling and PCA on the same training cosmologies, measures PCA-only reconstruction on the holdout set, trains an MLP with the same architecture, then measures the NN-predicted table and spectrum errors.

The PCA-only metric is especially useful: it shows the irreducible reconstruction loss at a given retained dimension before asking the NN to predict the PCA coordinates.

In [ ]:
TARGETS = {
    "P0/P2/P4": (Y_024, [0, 2, 4]),
    "P0/P2": (Y_02, [0, 2]),
}

results = []
fitted_models = {}

for target_name, (Y_all, multipoles) in TARGETS.items():
    print("\n" + "=" * 72)
    print(f"Target: {target_name}; table shape: {Y_all.shape[1:]}")
    print("=" * 72)

    Y_train_table = Y_all[train_idx]
    Y_valid_table = Y_all[valid_idx]

    component_scale = np.sqrt(
        np.mean(Y_train_table**2, axis=(0, 1, 2))
    )
    component_scale = np.maximum(component_scale, 1e-30)

    def flatten_scaled(table_array):
        return (
            table_array / component_scale[None, None, None, :]
        ).reshape(len(table_array), -1)

    Y_train_flat = flatten_scaled(Y_train_table)
    Y_valid_flat = flatten_scaled(Y_valid_table)

    max_allowed = min(len(train_idx) - 1, Y_train_flat.shape[1])
    for n_components in PCA_DIMENSIONS:
        if n_components > max_allowed:
            print(f"Skipping {n_components}: maximum allowed is {max_allowed}")
            continue

        print(f"\n--- {target_name}: {n_components} PCA components ---")
        pca = PCA(n_components=n_components, svd_solver="full")
        Z_train = pca.fit_transform(Y_train_flat)
        Z_valid = pca.transform(Y_valid_flat)

        # PCA-only reconstruction: no neural network involved.
        Y_valid_pca_flat = pca.inverse_transform(Z_valid)
        Y_valid_pca = (
            Y_valid_pca_flat.reshape(Y_valid_table.shape)
            * component_scale[None, None, None, :]
        )
        pca_metrics = table_error_summary(Y_valid_table, Y_valid_pca)

        # Same MLP architecture, optimizer, seed, and split for every run.
        mlp = MLPRegressor(**MLP_KWARGS)
        fit_start = time.perf_counter()
        mlp.fit(X_train, Z_train)
        fit_seconds = time.perf_counter() - fit_start

        Z_valid_pred = mlp.predict(X_valid)
        Y_valid_nn_flat = pca.inverse_transform(Z_valid_pred)
        Y_valid_nn = (
            Y_valid_nn_flat.reshape(Y_valid_table.shape)
            * component_scale[None, None, None, :]
        )
        nn_metrics = table_error_summary(Y_valid_table, Y_valid_nn)

        # Apply the same analytic standard-basis contraction to true and
        # predicted holdout tables. This isolates emulator error from theory setup.
        pk_valid_true = spectra_from_tables(Y_valid_table)
        pk_valid_pred = spectra_from_tables(Y_valid_nn)
        pk_metrics = table_error_summary(pk_valid_true, pk_valid_pred)

        row = {
            "target": target_name,
            "n_components": n_components,
            "pca_variance_explained": float(np.sum(pca.explained_variance_ratio_)),
            "pca_table_relative_rms": pca_metrics["relative_rms"],
            "nn_table_relative_rms": nn_metrics["relative_rms"],
            "nn_spectrum_relative_rms": pk_metrics["relative_rms"],
            "nn_spectrum_rmse": pk_metrics["rmse"],
            "fit_seconds": fit_seconds,
            "n_iter": int(mlp.n_iter_),
        }
        results.append(row)
        fitted_models[(target_name, n_components)] = {
            "pca": pca,
            "mlp": mlp,
            "component_scale": component_scale.copy(),
            "Y_valid_pred": Y_valid_nn,
            "pk_valid_true": pk_valid_true,
            "pk_valid_pred": pk_valid_pred,
        }

        print(
            f"variance={row['pca_variance_explained']:.6f} | "
            f"PCA table rel.RMS={row['pca_table_relative_rms']:.6e} | "
            f"NN table rel.RMS={row['nn_table_relative_rms']:.6e} | "
            f"NN spectrum rel.RMS={row['nn_spectrum_relative_rms']:.6e} | "
            f"fit={fit_seconds:.1f}s, iterations={mlp.n_iter_}"
        )

results_df = pd.DataFrame(results).sort_values(
    ["target", "n_components"]
).reset_index(drop=True)

print("\nCompleted configurations:", len(results_df))
display(results_df)

## 5. Compare PCA compression loss and NN prediction error

The first plot isolates PCA truncation error. The second includes the neural-network mapping. A gap between the two curves indicates error introduced by the NN beyond the information discarded by PCA.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for target_name, group in results_df.groupby("target"):
    group = group.sort_values("n_components")
    axes[0].plot(
        group["n_components"], group["pca_table_relative_rms"],
        marker="o", label=target_name,
    )
    axes[1].plot(
        group["n_components"], group["nn_table_relative_rms"],
        marker="o", label=target_name,
    )

axes[0].set_title("PCA-only holdout reconstruction")
axes[0].set_ylabel("Relative RMS PT-table error")
axes[1].set_title("PCA + neural-network holdout prediction")
axes[1].set_ylabel("Relative RMS PT-table error")

for ax in axes:
    ax.set_xlabel("Number of PCA components")
    ax.set_xticks(PCA_DIMENSIONS)
    ax.grid(alpha=0.25)
    ax.legend()

fig.tight_layout()
plt.show()

## 6. Compare final reconstructed-spectrum errors

This is the most directly relevant metric for the eventual emulator. Each held-out true table and its NN prediction are contracted with the same fiducial standard-basis nuisance vector. P0/P2/P4 reports the combined error across three multipoles; P0/P2 reports the combined error across two.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for target_name, group in results_df.groupby("target"):
    group = group.sort_values("n_components")
    ax.plot(
        group["n_components"], group["nn_spectrum_relative_rms"],
        marker="o", label=target_name,
    )

ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Relative RMS reconstructed-spectrum error")
ax.set_title("Holdout spectrum error versus PCA dimension")
ax.set_xticks(PCA_DIMENSIONS)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

# Also show per-multipole spectrum errors for every configuration.
per_ell_rows = []
for (target_name, n_components), model in fitted_models.items():
    true_pk = model["pk_valid_true"]
    pred_pk = model["pk_valid_pred"]
    for i, ell in enumerate(TARGETS[target_name][1]):
        err = table_error_summary(true_pk[:, i, :], pred_pk[:, i, :])
        per_ell_rows.append({
            "target": target_name,
            "n_components": n_components,
            "ell": ell,
            "spectrum_relative_rms": err["relative_rms"],
        })

per_ell_df = pd.DataFrame(per_ell_rows)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, ell in zip(axes, [0, 2, 4]):
    subset = per_ell_df[per_ell_df["ell"] == ell]
    for target_name, group in subset.groupby("target"):
        group = group.sort_values("n_components")
        ax.plot(
            group["n_components"], group["spectrum_relative_rms"],
            marker="o", label=target_name,
        )
    ax.set_title(f"ell={ell}")
    ax.set_xlabel("PCA components")
    ax.set_xticks(PCA_DIMENSIONS)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Relative RMS spectrum error")
axes[-1].legend()
fig.tight_layout()
plt.show()

## 7. Inspect cumulative PCA variance

Variance explained is a useful compression diagnostic, but it is not the selection criterion by itself. A low-variance direction can still matter to the final contracted spectra, so use the spectrum-error curves above when choosing a dimension.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for target_name, group in results_df.groupby("target"):
    group = group.sort_values("n_components")
    ax.plot(
        group["n_components"], group["pca_variance_explained"],
        marker="o", label=target_name,
    )
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative variance explained")
ax.set_title("PCA variance retained")
ax.set_xticks(PCA_DIMENSIONS)
ax.set_ylim(0, 1.01)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## 8. Save results for later comparison

The summary CSV can be downloaded from the Jupyter file browser or compared with future runs. The notebook also prints the full table for convenient copying into research notes.

In [ ]:
RESULTS_PATH = Path("pca_dimension_comparison_results.csv")
results_df.to_csv(RESULTS_PATH, index=False)
print("Saved:", RESULTS_PATH.resolve())
display(results_df.pivot(
    index="n_components",
    columns="target",
    values=[
        "pca_table_relative_rms",
        "nn_table_relative_rms",
        "nn_spectrum_relative_rms",
        "pca_variance_explained",
    ],
))

## Interpretation notes

- **PCA-only error** measures information lost by truncating the coefficient-table representation.
- **NN table error minus PCA-only error** is not an exact additive decomposition, but the separation of the two curves shows whether the NN is struggling beyond the PCA compression limit.
- **Spectrum error** is the primary practical metric because it measures the impact after the exact analytic nuisance contraction.
- Compare P0/P2/P4 against P0/P2 at the same component count, but also compare each family's best-performing component count.
- Do not choose a PCA dimension solely from cumulative variance explained.
- The same single holdout split makes this a controlled first comparison. If the curves are close or noisy, repeat with several split seeds before drawing strong conclusions.

The MLP uses the same 128x128 architecture and fixed optimizer settings in all 12 configurations. This notebook is a diagnostic sweep, not a final refit/export notebook.